# 📈 FolioTrack — Quickstart

This notebook demonstrates the core workflow of the `foliotrack` library:
1. Create a portfolio and buy securities
2. Update live prices from the market
3. Set target allocations
4. Run the MIQP optimizer to find the best trades
5. Backtest the portfolio over historical data
6. Save and load the portfolio

## 1. Setup

In [1]:
import logging
from foliotrack.domain.Portfolio import Portfolio
from foliotrack.services.MarketService import MarketService
from foliotrack.services.OptimizationService import OptimizationService
from foliotrack.services.BacktestService import BacktestService
from foliotrack.storage.PortfolioRepository import PortfolioRepository

logging.basicConfig(level=logging.INFO)

# Instantiate services
repo = PortfolioRepository()
optimizer = OptimizationService()
backtester = BacktestService()
market_service = MarketService()

## 2. Create Portfolio & Buy Securities

In [2]:
portfolio = Portfolio("Example Portfolio", currency="EUR")

# Buy ETFs — all affordable (€15-45 range)
portfolio.buy_security("LEM.PA", volume=10.0, price=17.0, date="2024-01-15", fill=True)   # Amundi Emerging Markets ~€17
portfolio.buy_security("CW8.PA", volume=10.0, price=30.0, date="2024-01-15", fill=True)   # Amundi MSCI World ~€30
portfolio.buy_security("PANX.PA", volume=5.0, price=45.0, date="2024-03-01", fill=True)   # Amundi Nasdaq-100 ~€45

INFO:root:Security 'LEM.PA' added to portfolio with volume 10.0.
INFO:root:Security 'CW8.PA' added to portfolio with volume 10.0.
INFO:root:Security 'PANX.PA' added to portfolio with volume 5.0.


## 3. Update Prices from Market

In [3]:
market_service.update_prices(portfolio)

# Display current portfolio state
for info in portfolio.get_portfolio_info():
    print(f"{info['name']:50s}  {info['ticker']:10s}  Price: {info['price_in_security_currency']:>8.2f}{info['symbol']}  Volume: {info['volume']:.0f}")

Amundi MSCI Emerging Markets Swap II UCITS ETF EUR Acc  LEM.PA      Price:    16.75€  Volume: 10
Amundi Index Solutions - Amundi MSCI World Swap UCITS ETF EUR Acc  CW8.PA      Price:   615.00€  Volume: 10
Amundi PEA US Tech Screened UCITS ETF - Acc         PANX.PA     Price:    65.18€  Volume: 5


## 4. Set Target Allocations & Optimize

In [4]:
# Target allocation: 50% World, 30% Emerging, 20% Nasdaq
portfolio.set_target_share("CW8.PA", 0.5)
portfolio.set_target_share("LEM.PA", 0.3)
portfolio.set_target_share("PANX.PA", 0.2)

In [5]:
# Optimize: invest €500 with at least 99% utilization
security_counts, total_to_invest, final_shares = optimizer.solve_equilibrium(
    portfolio,
    investment_amount=500.0,
    min_percent_to_invest=0.99,
)

print(f"\nTotal to invest: {total_to_invest:.2f}€")
print("\nRecommended trades:")
for info in portfolio.get_portfolio_info():
    print(f"  {info['ticker']:10s}  Buy: {info['volume_to_buy']:>3} units  ({info['amount_to_invest']:>8.2f}€)  Final share: {info['final_share']:.2%}")

INFO:root:Optimisation status: optimal
INFO:root:Number of each Security to buy:
INFO:root:  Amundi MSCI Emerging Markets Swap II UCITS ETF EUR Acc: 22 units
INFO:root:  Amundi Index Solutions - Amundi MSCI World Swap UCITS ETF EUR Acc: 0 units
INFO:root:  Amundi PEA US Tech Screened UCITS ETF - Acc: 2 units
INFO:root:Amount to spend and final share of each Security:
INFO:root:  Amundi MSCI Emerging Markets Swap II UCITS ETF EUR Acc: 368.50€, Final share = 0.0750
INFO:root:  Amundi Index Solutions - Amundi MSCI World Swap UCITS ETF EUR Acc: 0.00€, Final share = 0.8611
INFO:root:  Amundi PEA US Tech Screened UCITS ETF - Acc: 130.36€, Final share = 0.0639
INFO:root:Total amount to invest: 498.86€



Total to invest: 498.86€

Recommended trades:
  LEM.PA      Buy:  22 units  (  368.50€)  Final share: 7.50%
  CW8.PA      Buy:   0 units  (    0.00€)  Final share: 86.11%
  PANX.PA     Buy:   2 units  (  130.36€)  Final share: 6.39%


## 5. Backtest

In [6]:
result = backtester.run_backtest(
    portfolio, market_service, start_date="2020-01-01", end_date="2025-01-01"
)
result.display()

[*********************100%***********************]  3 of 3 completed


Stat                 Example Portfolio
-------------------  -------------------
Start                2020-01-01
End                  2026-04-10
Risk-free rate       0.00%

Total Return         97.48%
Daily Sharpe         0.71
Daily Sortino        0.97
CAGR                 11.46%
Max Drawdown         -31.70%
Calmar Ratio         0.36

MTD                  5.83%
3m                   0.44%
6m                   7.62%
YTD                  3.30%
1Y                   35.15%
3Y (ann.)            17.26%
5Y (ann.)            10.21%
10Y (ann.)           -
Since Incep. (ann.)  11.46%

Daily Sharpe         0.71
Daily Sortino        0.97
Daily Mean (ann.)    12.14%
Daily Vol (ann.)     17.09%
Daily Skew           -0.78
Daily Kurt           7.93
Best Day             7.69%
Worst Day            -8.96%

Monthly Sharpe       0.87
Monthly Sortino      1.41
Monthly Mean (ann.)  12.08%
Monthly Vol (ann.)   13.85%
Monthly Skew         -0.44
Monthly Kurt         0.08
Best Month           9.54%
Worst Month    

## 6. Save & Load Portfolio

In [7]:
# Save
repo.save_to_json(portfolio, "../Portfolios/investment_example.json")

# Load it back
loaded = repo.load_from_json("../Portfolios/investment_example.json")
print(f"Loaded portfolio: {loaded.name}, {len(loaded.securities)} securities")

INFO:root:Portfolio saved to ../Portfolios/investment_example.json


Loaded portfolio: Example Portfolio, 3 securities


## 7. Interactive Dashboard

For a full visual experience, launch the Streamlit dashboard:

```bash
pip install foliotrack[dashboard]
dash
```